# 02 · Error analysis

Where does the champion model go wrong, and what does that mean for a fraud-ops team?
* **False positives**: high scores on legitimate customers. This is where friction lands.
* **Missed fraud**: fraud the recommended policy approved. Card testing? A product line? New accounts?

In [ ]:
import os
import altair as alt
import numpy as np
import pandas as pd

from fraud import charts, db  # charts registers the project's Altair theme
from fraud.config import get_paths
from fraud.io import load_json

PROFILE = os.environ.get("FRAUD_PROFILE", "full")  # "sample" = synthetic data
paths = get_paths(PROFILE)
pd.set_option("display.max_columns", 50, "display.width", 160)
print("profile:", PROFILE)

In [ ]:
res = load_json(paths.reports / "train_results.json")
champ = res["champion"]
scored = pd.read_parquet(paths.scored)
test = scored[scored.split == "test"].copy()
dec = pd.read_parquet(paths.processed / "decisions.parquet")
test = test.merge(dec, on="TransactionID")
p = f"p_{champ}"
test["amount_band"] = pd.cut(test.TransactionAmt, [0, 10, 50, 200, 1000, 1e9], labels=["<$10", "$10-50", "$50-200", "$200-1k", "$1k+"])
print(champ, len(test), "test rows")

## 1. Top 20 false positives

In [ ]:
cols = ["TransactionID", p, "TransactionAmt", "ProductCD", "has_identity", "is_first_txn_uid", "card_tx_1h",
        "amt_vs_uid_mean", "email_match", "device_family", "p_email_provider", "decision"]
fp = test[test.isFraud == 0].nlargest(20, p)[cols]
fp

In [ ]:
def profile(df, label):
    return pd.Series({
        "n": len(df), "first txn on account": df.is_first_txn_uid.mean(), "has identity": df.has_identity.mean(),
        "median amount": df.TransactionAmt.median(), "card used in last hour": (df.card_tx_1h > 0).mean(),
        "ProductCD=W": (df.ProductCD == "W").mean(), "ProductCD=C": (df.ProductCD == "C").mean(),
    }, name=label)

legit = test[test.isFraud == 0]
top_legit = legit[legit[p] >= legit[p].quantile(0.99)]
pd.concat([profile(legit, "all legitimate"), profile(top_legit, "top 1% scored legitimate"),
           profile(test[test.isFraud == 1], "all fraud")], axis=1).round(3)

**Where friction lands (DRAFT):** _the false positives skew toward ... (new accounts? no identity? product C?)._

## 2. Fraud the recommended policy approved

In [ ]:
missed = test[(test.isFraud == 1) & (test.decision == "approve")]
caught = test[(test.isFraud == 1) & (test.decision != "approve")]
print(f"missed {len(missed)} of {int(test.isFraud.sum())} frauds; ${missed.TransactionAmt.sum():,.0f} of ${test.TransactionAmt[test.isFraud == 1].sum():,.0f}")
pd.concat([missed.amount_band.value_counts(normalize=True).rename("missed"),
           caught.amount_band.value_counts(normalize=True).rename("caught")], axis=1).round(3)

In [ ]:
pd.concat([missed.ProductCD.value_counts(normalize=True).rename("missed"),
           caught.ProductCD.value_counts(normalize=True).rename("caught")], axis=1).round(3)

In [ ]:
# card-testing check: small amounts on a card already busy in the last hour
for name, d in (("missed", missed), ("caught", caught)):
    small = d.TransactionAmt < 10
    print(f"{name:7s} small(<$10): {small.mean():.1%}   small & card used in last hour: {(small & (d.card_tx_1h > 0)).mean():.1%}")

**Missed-fraud pattern (DRAFT):** _most missed fraud is small-dollar ..., which the cost policy approves on purpose: declining a $5 transaction risks $5 of friction to save ~$20._

## 3. Cold start: accounts never seen before the test window

In [ ]:
rows = []
for k, v in res["experiments"].items():
    if "test_new_uid" in v:
        rows.append({"model": k, "all test PR-AUC": v["test"]["pr_auc"], "new-account PR-AUC": v["test_new_uid"]["pr_auc"],
                     "new-account base rate": v["test_new_uid"]["base_rate"]})
print(f"{res['new_uid_test_share']:.1%} of test rows are new accounts")
pd.DataFrame(rows).round(4)

## 4. Reason codes for example transactions

In [ ]:
ex = load_json(paths.reports / "explain.json")["examples"]
for e in ex:
    print(f"{e['type']:15s} id={e['TransactionID']}  p={e['p']:.3f}")
    for r in e["reasons"]:
        print("    -", r["text"])